# Privacy Analysis

Evaluates privacy of synthetic and reconstructed SNP data against real training data.

**Metrics computed:**
1. **IMR** — Identical Match Rate (exact copy detection)
2. **DCR** — Distance to Closest Record (min distance distribution)
3. **NNAA** — Nearest Neighbor Adversarial Accuracy
4. **MI** — Distance-based Membership Inference (ROC-AUC)
5. **NNDR** — Nearest Neighbor Distance Ratio (copying detection)
6. **MAF** — Allele Frequency Comparison (fidelity sanity check)

Results are saved incrementally to checkpoint directories to survive crashes.

## 1. Setup & Configuration

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

# Add project root to path
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from snpgen.evaluation.privacy import (
    PrivacyEvaluator,
    load_privacy_data_from_checkpoint,
    load_privacy_data_manual,
    plot_dcr_distributions,
    plot_nnaa_summary,
    plot_mi_roc,
    plot_nndr_histogram,
    plot_maf_scatter,
    plot_privacy_summary_table,
)

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# ============================================================
# USER SETTINGS — Edit these paths before running
# ============================================================

# Base directory containing all SNPgen checkpoints.
# All TRAITS entries below are resolved relative to this path.
CKPT_BASE = '/path/to/snpgen/checkpoints'

# ============================================================
# EVALUATION SETTINGS
# ============================================================
DISTANCE = "hamming"       # "hamming" or "manhattan"
PER_CLASS = True           # Compute metrics per label class
DEVICE = "auto"            # "auto", "gpu", or "cpu"
EVALUATE_SYNTHETIC = True
EVALUATE_RECONSTRUCTED = True

In [ ]:
# ============================================================
# TRAIT CONFIGURATION
# ============================================================
# Define all traits to evaluate. Each entry needs:
#   - ddpm_checkpoint: path to DDPM checkpoint directory
# ============================================================

TRAITS = {
    "TRAIT1": {
        "ddpm_checkpoint": f"{CKPT_BASE}/trait1/trait1_ddpm_emb128_small_white-31645617/",
    },
}

## 2. Data Loading

In [ ]:
bundles = {}

for trait_name, cfg in TRAITS.items():
    print(f"\n{'='*60}")
    print(f" Loading: {trait_name}")
    print(f"{'='*60}")
    
    try:
        if cfg.get("manual", False):
            bundle = load_privacy_data_manual(
                dataset_path=cfg["dataset_path"],
                ddpm_checkpoint_dir=cfg["ddpm_checkpoint"],
                vae_checkpoint_dir=cfg.get("vae_checkpoint", ""),
                model_name=trait_name,
                seed=cfg.get("seed", 42),
                val_ratio=cfg.get("val_ratio", 0.2),
                test_ratio=cfg.get("test_ratio", 0.1),
            )
        else:
            bundle = load_privacy_data_from_checkpoint(
                ddpm_checkpoint_dir=cfg["ddpm_checkpoint"],
                model_name=trait_name,
            )
        
        bundle.summary()
        bundles[trait_name] = bundle
        
    except Exception as e:
        print(f"  ERROR loading {trait_name}: {e}")
        continue

print(f"\nLoaded {len(bundles)}/{len(TRAITS)} traits successfully.")

## 3. Privacy Evaluation

In [ ]:
evaluator = PrivacyEvaluator(
    distance=DISTANCE,
    per_class=PER_CLASS,
    device=DEVICE,
)

all_results = {}

for trait_name, bundle in bundles.items():
    
    # ── Synthetic privacy (saved to DDPM checkpoint dir) ──
    if EVALUATE_SYNTHETIC:
        syn_output_dir = os.path.join(bundle.ddpm_checkpoint_dir, "privacy_results")
        print(f"\n{'#'*60}")
        print(f"# {trait_name} — SYNTHETIC")
        print(f"# Output: {syn_output_dir}")
        print(f"{'#'*60}")
        
        syn_results = evaluator.evaluate(
            bundle,
            output_dir=syn_output_dir,
            eval_target='synthetic',
        )
        all_results[f"{trait_name}_syn"] = syn_results
    
    # ── Reconstruction privacy (saved to VAE checkpoint dir) ──
    if EVALUATE_RECONSTRUCTED and bundle.reconstructed is not None:
        recon_output_dir = os.path.join(bundle.vae_checkpoint_dir, "privacy_results")
        print(f"\n{'#'*60}")
        print(f"# {trait_name} — RECONSTRUCTED")
        print(f"# Output: {recon_output_dir}")
        print(f"{'#'*60}")
        
        recon_results = evaluator.evaluate(
            bundle,
            output_dir=recon_output_dir,
            eval_target='reconstructed',
        )
        all_results[f"{trait_name}_recon"] = recon_results
    elif EVALUATE_RECONSTRUCTED:
        print(f"\n  {trait_name}: No reconstructed data available, skipping.")

print(f"\nEvaluation complete. {len(all_results)} result sets computed.")

## 4. Visualization

In [ ]:
for trait_key, results in all_results.items():
    print(f"\n{'='*60}")
    print(f" {trait_key}")
    print(f"{'='*60}")
    
    # DCR
    dcr_r = results.get('dcr__overall')
    if dcr_r is not None:
        fig = plot_dcr_distributions(dcr_r, title=f'{trait_key} — DCR Distribution')
        plt.show()
    
    # NNAA
    nnaa_r = results.get('nnaa__overall')
    if nnaa_r is not None:
        fig = plot_nnaa_summary(nnaa_r, title=f'{trait_key} — NNAA')
        plt.show()
    
    # MI
    mi_r = results.get('mi__overall')
    if mi_r is not None:
        fig = plot_mi_roc(mi_r, title=f'{trait_key} — Membership Inference')
        plt.show()
    
    # NNDR
    nndr_r = results.get('nndr__overall')
    if nndr_r is not None:
        fig = plot_nndr_histogram(nndr_r, title=f'{trait_key} — NNDR')
        plt.show()
    
    # MAF
    maf_r = results.get('maf__overall')
    if maf_r is not None:
        fig = plot_maf_scatter(maf_r, title=f'{trait_key} — MAF')
        plt.show()

### Per-class visualizations

In [ ]:
if PER_CLASS:
    for trait_key, results in all_results.items():
        # Dynamically discover per-class suffixes from result keys
        class_keys = sorted([k for k in results if k.startswith('dcr__class_')])
        if not class_keys:
            continue

        for dcr_key in class_keys:
            suffix = dcr_key.replace('dcr__', '')
            label_val = suffix.replace('class_', '')
            label_name = f'Class {label_val}'

            dcr_r = results.get(f'dcr__{suffix}')
            if dcr_r is None:
                continue

            print(f"\n--- {trait_key} | {label_name} (label={label_val}) ---")

            fig = plot_dcr_distributions(dcr_r, title=f'{trait_key} — DCR ({label_name})')
            plt.show()

            nnaa_r = results.get(f'nnaa__{suffix}')
            if nnaa_r is not None:
                fig = plot_nnaa_summary(nnaa_r, title=f'{trait_key} — NNAA ({label_name})')
                plt.show()

## 5. Cross-Trait Summary

In [ ]:
fig = plot_privacy_summary_table(all_results)
if fig is not None:
    plt.show()